# Experiment: CytoRAG Embedding Models and Chunking Strategies for Russian Medical Retrieval

Objective:
- Reuse the main CytoRAG logic from `test.ipynb`: hybrid retrieval (`BM25 + dense embeddings + cross-encoder rerank`), optional local `vLLM`, manual RAG metrics, and `ragas` metrics.
- Compare embedding models that make sense for Russian corpora and the medical domain.
- Add a dedicated chunking study inside the same notebook to test which chunk construction strategy better fits short but sometimes multi-part Bethesda cytology cases.
- Keep the experiment runnable on a single NVIDIA T4 by loading only one embedding model on GPU at a time and moving re-ranking / `ragas` embeddings to CPU.

Success criteria:
- Each embedding candidate can be evaluated sequentially on T4 without keeping multiple large models in memory.
- Chunking candidates can be evaluated with a fixed embedding backbone so the effect of document segmentation is isolated.
- The output is a compact comparison table over the metrics already used in the project: `hit_rate`, `context_relevance`, `faithfulness`, `answer_correctness`, `context_precision`, `context_recall`, `answer_relevancy`.
- The notebook saves markdown-ready tables and a concise report that can be pasted into a presentation or `.md` document.


## Plan

Hypotheses:
- `DeepPavlov/rubert-base-cased-sentence` should be a strong Russian-only baseline for short clinical-style cases.
- `BAAI/bge-m3` may win on multilingual retrieval robustness, especially in the hybrid setup.
- `FremyCompany/BioLORD-2023-M` is an interesting biomedical probe, but it is not Russian-native, so it tests domain strength against language mismatch.
- `microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext` is an English biomedical control and may underperform on Russian text, but it is still useful as a domain-only baseline.
- The current project baseline `cointegrated/rubert-tiny2` should stay in the sweep, because it is the closest reference to `test.ipynb`.
- For chunking, structure-aware splitting by subcase or lobe should outperform naive fixed windows because the dataset contains short records with occasional multi-part findings inside one field.

What is evaluated:
- Retrieval stage: whether the gold case is present in the top retrieved contexts.
- Optional generation stage: manual LLM-as-a-judge metrics reused from `test.ipynb`.
- Optional `ragas` stage: `context_precision`, `context_recall`, `faithfulness`, `answer_relevancy`.
- Chunking stage: whether structural and sentence-window strategies improve retrieval and `ragas` quality relative to whole-case indexing.

Important note on this dataset:
- `bethesda_ground_truth.json` contains the same generic question for all rows, so the notebook derives a case-specific retrieval query from the beginning of each cytology description.
- Because the real corpus does not explicitly contain the final Bethesda label in the retrieved context, `hit_rate` is adapted to mean: "the gold case was recovered in the final top-k contexts".
- Some rows contain multiple subcases in a single `simulated_context`, so the chunking study tracks hits by parent `case_id`, not only by exact chunk text.


In [ ]:
import importlib.util
import socket
import subprocess
import sys
import time
from typing import Iterable, List

MODULE_INSTALL_STATUS = {}


def module_available(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is not None


def pip_network_ready(hosts: Iterable[str] = ("pypi.org", "files.pythonhosted.org")) -> bool:
    for host in hosts:
        try:
            socket.gethostbyname(host)
            return True
        except OSError:
            continue
    return False


def pip_install_if_needed(
    packages: List[str],
    required_modules: List[str],
    label: str,
    optional: bool = False,
) -> bool:
    missing_modules = [module for module in required_modules if not module_available(module)]
    if not missing_modules:
        MODULE_INSTALL_STATUS[label] = "already_available"
        print(f"{label}: already available, pip install skipped.")
        return True

    if not pip_network_ready():
        MODULE_INSTALL_STATUS[label] = "skipped_no_network"
        print(f"{label}: skipped because PyPI DNS/network is unavailable.")
        print(f"  Missing modules: {missing_modules}")
        if optional:
            print("  This dependency group is optional for this notebook path.")
        return False

    started_at = time.time()
    cmd = [sys.executable, "-m", "pip", "install", "-q", "-U", *packages]
    print(f"{label}: installing {packages}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    duration = time.time() - started_at
    if result.returncode == 0:
        MODULE_INSTALL_STATUS[label] = "installed"
        print(f"{label}: installed successfully in {duration:.1f}s.")
        return True

    MODULE_INSTALL_STATUS[label] = "failed"
    print(f"{label}: pip install failed after {duration:.1f}s with exit code {result.returncode}.")
    stderr_tail = "\n".join(line for line in result.stderr.strip().splitlines()[-12:] if line.strip())
    stdout_tail = "\n".join(line for line in result.stdout.strip().splitlines()[-12:] if line.strip())
    if stderr_tail:
        print("  stderr tail:")
        print(stderr_tail)
    if stdout_tail:
        print("  stdout tail:")
        print(stdout_tail)
    if optional:
        print("  Continuing without this optional dependency group.")
        return False
    return False


BASE_DEPENDENCIES_READY = pip_install_if_needed(
    packages=[
        "transformers>=4.55.2,<5",
        "tokenizers>=0.21.1,<0.22",
        "sentence-transformers>=3.0,<4",
        "faiss-cpu>=1.8,<2",
        "rank-bm25>=0.2.2",
        "python-dotenv>=1.0",
        "accelerate>=0.30",
        "requests==2.32.4",
    ],
    required_modules=[
        "transformers",
        "tokenizers",
        "sentence_transformers",
        "faiss",
        "rank_bm25",
        "dotenv",
        "accelerate",
        "requests",
    ],
    label="Base retrieval dependencies",
)

VLLM_DEPENDENCIES_READY = pip_install_if_needed(
    packages=["vllm==0.11.0"],
    required_modules=["vllm"],
    label="Optional local vLLM dependency",
    optional=True,
)

print(
    {
        "base_dependencies_ready": BASE_DEPENDENCIES_READY,
        "vllm_dependencies_ready": VLLM_DEPENDENCIES_READY,
        "module_install_status": MODULE_INSTALL_STATUS,
    }
)


In [ ]:
import numpy as np
import pandas as pd
import requests
import sentence_transformers
import transformers

print({
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "requests": requests.__version__,
    "sentence_transformers": sentence_transformers.__version__,
    "transformers": transformers.__version__,
})


In [ ]:
from __future__ import annotations

import gc
import json
import os
import random
import re
import subprocess
import threading
import time
import warnings
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import requests
import torch
from dotenv import load_dotenv
from sentence_transformers import CrossEncoder, SentenceTransformer

try:
    import faiss
    FAISS_AVAILABLE = True
except Exception:
    faiss = None
    FAISS_AVAILABLE = False

try:
    from rank_bm25 import BM25Okapi
    BM25_AVAILABLE = True
except Exception:
    BM25Okapi = None
    BM25_AVAILABLE = False

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
warnings.filterwarnings("ignore")

load_dotenv()


class SimpleTokenOverlapRetriever:
    def __init__(self, tokenized_corpus: List[List[str]], k1: float = 1.5, b: float = 0.75):
        self.tokenized_corpus = tokenized_corpus
        self.k1 = k1
        self.b = b
        self.corpus_size = len(tokenized_corpus)
        self.avgdl = sum(len(doc) for doc in tokenized_corpus) / max(self.corpus_size, 1)
        self.doc_freq = Counter()
        for doc in tokenized_corpus:
            for token in set(doc):
                self.doc_freq[token] += 1

    def get_scores(self, query_tokens: List[str]) -> np.ndarray:
        scores = np.zeros(self.corpus_size, dtype="float32")
        if not query_tokens:
            return scores

        query_counter = Counter(query_tokens)
        avgdl = self.avgdl if self.avgdl > 0 else 1.0
        for idx, doc_tokens in enumerate(self.tokenized_corpus):
            if not doc_tokens:
                continue
            doc_counter = Counter(doc_tokens)
            doc_len = len(doc_tokens)
            score = 0.0
            for token, _ in query_counter.items():
                tf = doc_counter.get(token, 0)
                if tf == 0:
                    continue
                df = self.doc_freq.get(token, 0)
                idf = np.log(1.0 + (self.corpus_size - df + 0.5) / (df + 0.5))
                norm = self.k1 * (1.0 - self.b + self.b * doc_len / avgdl)
                score += idf * ((tf * (self.k1 + 1.0)) / (tf + norm))
            scores[idx] = score
        return scores


class NumpyInnerProductIndex:
    def __init__(self, dimension: int):
        self.dimension = dimension
        self.embeddings = np.empty((0, dimension), dtype="float32")

    def add(self, embeddings: np.ndarray) -> None:
        self.embeddings = embeddings.astype("float32")

    def search(self, query_embedding: np.ndarray, top_k: int) -> Tuple[np.ndarray, np.ndarray]:
        if self.embeddings.size == 0:
            return np.empty((1, 0), dtype="float32"), np.empty((1, 0), dtype="int64")
        scores = query_embedding @ self.embeddings.T
        top_k = min(top_k, self.embeddings.shape[0])
        sorted_idx = np.argsort(scores[0])[::-1][:top_k]
        return scores[:, sorted_idx], sorted_idx.reshape(1, -1)


def build_lexical_retriever(tokenized_corpus: List[List[str]]) -> Tuple[Any, str]:
    if BM25_AVAILABLE and BM25Okapi is not None:
        return BM25Okapi(tokenized_corpus), "rank_bm25"
    return SimpleTokenOverlapRetriever(tokenized_corpus), "simple_overlap_fallback"


def build_dense_index(embeddings: np.ndarray) -> Tuple[Any, str]:
    if FAISS_AVAILABLE and faiss is not None:
        index = faiss.IndexFlatIP(embeddings.shape[1])
        index.add(embeddings)
        return index, "faiss"
    index = NumpyInnerProductIndex(embeddings.shape[1])
    index.add(embeddings)
    return index, "numpy_fallback"


def resolve_base_dir() -> Path:
    cwd = Path.cwd()
    if (cwd / "bethesda_ground_truth.json").exists():
        return cwd
    candidate = cwd / "CytoRAG"
    if (candidate / "bethesda_ground_truth.json").exists():
        return candidate
    raise FileNotFoundError("Не удалось найти bethesda_ground_truth.json ни в текущей директории, ни в ./CytoRAG")


BASE_DIR = resolve_base_dir()
DATA_PATH = BASE_DIR / "bethesda_ground_truth.json"
EXISTING_RAGAS_CSV = BASE_DIR / "ragas_evaluation_results.csv"
ARTIFACT_DIR = BASE_DIR / "artifacts" / "embedding_study"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RERANKER_DEVICE = "cpu"
RERANKER_NAME = "DiTy/cross-encoder-russian-msmarco"
USE_FP16 = EMBEDDING_DEVICE == "cuda"
DENSE_TOP_K = 5
FINAL_TOP_K = 3

RUN_RETRIEVAL_SWEEP = True
RUN_LLM_MANUAL_METRICS = False
RUN_RAGAS = True
START_LOCAL_VLLM = False

VLLM_MODEL_NAME = os.getenv("VLLM_MODEL_NAME", "Qwen/Qwen2.5-3B-Instruct")
EVAL_MODEL_NAME = os.getenv("EVAL_MODEL_NAME", VLLM_MODEL_NAME)
OPENAI_API_BASE = os.getenv("OPENAI_API_BASE", "http://localhost:9999/v1")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "EMPTY")

print(
    {
        "base_dir": str(BASE_DIR),
        "embedding_device": EMBEDDING_DEVICE,
        "reranker_device": RERANKER_DEVICE,
        "faiss_available": FAISS_AVAILABLE,
        "rank_bm25_available": BM25_AVAILABLE,
        "artifacts": str(ARTIFACT_DIR),
        "run_retrieval_sweep": RUN_RETRIEVAL_SWEEP,
        "run_llm_manual_metrics": RUN_LLM_MANUAL_METRICS,
        "run_ragas": RUN_RAGAS,
        "start_local_vllm": START_LOCAL_VLLM,
    }
)


In [ ]:
@dataclass
class ModelSpec:
    label: str
    model_name: str
    batch_size: int
    max_length: int
    notes: str


MODEL_SPECS: List[ModelSpec] = [
    ModelSpec(
        label="rubert_tiny2_baseline",
        model_name="cointegrated/rubert-tiny2",
        batch_size=32,
        max_length=384,
        notes="Current lightweight baseline closest to test.ipynb",
    ),
    ModelSpec(
        label="rubert_base_sentence",
        model_name="DeepPavlov/rubert-base-cased-sentence",
        batch_size=16,
        max_length=384,
        notes="Russian sentence encoder",
    ),
    ModelSpec(
        label="bge_m3",
        model_name="BAAI/bge-m3",
        batch_size=4,
        max_length=512,
        notes="Multilingual retrieval model; no query prefix needed for bge-m3",
    ),
    ModelSpec(
        label="biolord_2023_m",
        model_name="FremyCompany/BioLORD-2023-M",
        batch_size=8,
        max_length=512,
        notes="Multilingual biomedical encoder",
    ),
    ModelSpec(
        label="pubmedbert_fulltext",
        model_name="microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
        batch_size=8,
        max_length=384,
        notes="English biomedical control baseline",
    ),
]


def clear_torch_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def tokenize_ru_medical(text: str) -> List[str]:
    return re.findall(r"[A-Za-zА-Яа-яЁё0-9№/-]+", text.lower())


def extract_case_anchor(text: str, max_len: int = 110) -> str:
    first_sentence = re.split(r"(?<=[.!?])\s+", text.strip())[0]
    return first_sentence[:max_len].rstrip(" ,;:")


def extract_bethesda_label(text: str) -> Optional[str]:
    match = re.search(r"bethesda\s*[-–]\s*([ivx]+)", text, flags=re.IGNORECASE)
    return match.group(1).upper() if match else None


def build_eval_rows(raw_rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    eval_rows: List[Dict[str, Any]] = []
    for row in raw_rows:
        anchor = extract_case_anchor(row["simulated_context"])
        eval_rows.append(
            {
                "id": int(row["id"]),
                "original_query": row["query"],
                "retrieval_query": f"Определи диагностическую категорию Bethesda для клинического случая: {anchor}",
                "case_anchor": anchor,
                "simulated_context": row["simulated_context"],
                "ground_truth": row["ground_truth"],
                "bethesda_label": extract_bethesda_label(row["ground_truth"]),
            }
        )
    return eval_rows


def summarize_existing_ragas(csv_path: Path) -> pd.DataFrame:
    if not csv_path.exists():
        return pd.DataFrame()

    df = pd.read_csv(csv_path)
    summary = {"model": "existing_csv_baseline"}
    for metric in ["context_precision", "context_recall", "faithfulness", "answer_relevancy"]:
        if metric in df.columns:
            series = pd.to_numeric(df[metric], errors="coerce")
            summary[metric] = round(series.mean() * 100, 2) if series.notna().any() else np.nan
    return pd.DataFrame([summary])


raw_rows = json.loads(DATA_PATH.read_text(encoding="utf-8"))
eval_rows = build_eval_rows(raw_rows)
documents = [
    (row["simulated_context"], {"case_id": row["id"], "case_anchor": row["case_anchor"]})
    for row in eval_rows
]
existing_ragas_baseline_df = summarize_existing_ragas(EXISTING_RAGAS_CSV)

preview_df = pd.DataFrame(
    {
        "id": [row["id"] for row in eval_rows],
        "case_anchor": [row["case_anchor"] for row in eval_rows],
        "bethesda_label": [row["bethesda_label"] for row in eval_rows],
    }
)

print(f"Loaded {len(eval_rows)} Bethesda cases from {DATA_PATH.name}")
display(preview_df.head())
if not existing_ragas_baseline_df.empty:
    display(existing_ragas_baseline_df)


## Optional Local vLLM

This notebook now follows the same local evaluation pattern as `test.ipynb`:
- `vllm==0.11.0` is installed in the base dependencies cell together with compatible `transformers` and `tokenizers`.
- If `RUN_RAGAS=True` and `OPENAI_API_BASE` still points to `localhost`, the notebook can auto-start local `vLLM` for evaluation.

Important Colab note:
- In Google Colab with T4 GPU, this should work similarly to `test.ipynb`, but `vLLM` startup still takes time and may fail if GPU memory is exhausted.
- If a local endpoint still does not come up, you can either rerun after a runtime restart or provide an external OpenAI-compatible endpoint.

Recommended flows:
1. Full local evaluation in Colab or CUDA environment: keep `RUN_RAGAS=True`.
2. Retrieval-only run: set `RUN_RAGAS=False`.
3. External API evaluation: set `OPENAI_API_BASE`, `OPENAI_API_KEY`, and `EVAL_MODEL_NAME`.


In [ ]:
last_vllm_logs: List[str] = []


def explain_eval_skip(stage_name: str, api_base: Optional[str] = None) -> None:
    import importlib.util

    base = (api_base or OPENAI_API_BASE).rstrip("/")
    localhost_prefixes = (
        "http://localhost",
        "http://127.0.0.1",
        "https://localhost",
        "https://127.0.0.1",
    )
    points_to_localhost = base.startswith(localhost_prefixes)
    vllm_installed = importlib.util.find_spec("vllm") is not None
    endpoint_ready, detail = check_openai_compatible_endpoint(base, OPENAI_API_KEY)
    log_tail = "\\n".join(last_vllm_logs[-10:]) if last_vllm_logs else "(no vLLM logs captured)"

    print(f"{stage_name} skipped.")
    print(f"  RUN_RAGAS={RUN_RAGAS}")
    print(f"  OPENAI_API_BASE={base}")
    print(f"  EVAL_MODEL_NAME={EVAL_MODEL_NAME}")
    print(f"  running_in_colab={running_in_colab()}")
    print(f"  cuda_available={torch.cuda.is_available()}")
    print(f"  vllm_installed={vllm_installed}")
    print(f"  endpoint_ready={endpoint_ready}")
    if points_to_localhost:
        print("  reason: notebook is configured to use a localhost endpoint, but no local OpenAI-compatible endpoint is reachable.")
    else:
        print("  reason: configured OPENAI_API_BASE is not reachable.")
    print(f"  last_endpoint_error={detail or '(no detail)'}")
    print(f"  recent_vllm_logs={log_tail}")


def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore

        return True
    except Exception:
        return False


def can_attempt_local_vllm() -> bool:
    import importlib.util

    return torch.cuda.is_available() and importlib.util.find_spec("vllm") is not None


def launch_local_vllm(
    model_name: str = VLLM_MODEL_NAME,
    port: int = 9999,
    max_model_len: int = 2048,
    gpu_memory_utilization: float = 0.82,
):
    import importlib.util

    if not torch.cuda.is_available():
        raise RuntimeError("Для локального vLLM в этом ноутбуке нужен доступный CUDA GPU.")
    if importlib.util.find_spec("vllm") is None:
        raise ModuleNotFoundError(
            "Пакет vllm не установлен. Установите его в окружение ноутбука или укажите внешний OPENAI_API_BASE."
        )

    cmd = [
        "python",
        "-m",
        "vllm.entrypoints.openai.api_server",
        "--host",
        "0.0.0.0",
        "--port",
        str(port),
        "--model",
        model_name,
        "--max-model-len",
        str(max_model_len),
        "--dtype",
        "half",
        "--gpu-memory-utilization",
        str(gpu_memory_utilization),
        "--enforce-eager",
    ]

    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )

    def watch() -> None:
        while True:
            line = process.stdout.readline() if process.stdout else ""
            if not line and process.poll() is not None:
                break
            if line:
                clean = line.strip()
                last_vllm_logs.append(clean)
                if len(last_vllm_logs) > 20:
                    last_vllm_logs.pop(0)
                print(clean)

    watcher = threading.Thread(target=watch, daemon=True)
    watcher.start()

    health_url = f"http://localhost:{port}/health"
    for _ in range(300):
        try:
            response = requests.get(health_url, timeout=3)
            if response.status_code == 200:
                print(f"vLLM is ready at {health_url}")
                return process
        except requests.RequestException:
            pass
        time.sleep(1)

    raise RuntimeError("Не удалось дождаться запуска локального vLLM сервера.")


def check_openai_compatible_endpoint(api_base: str, api_key: str = OPENAI_API_KEY, timeout: int = 10) -> Tuple[bool, str]:
    base = api_base.rstrip("/")
    probe_urls = [base + "/models"]
    if base.endswith("/v1"):
        probe_urls.append(base[:-3] + "/health")

    headers = {}
    if api_key and api_key != "EMPTY":
        headers["Authorization"] = f"Bearer {api_key}"

    last_error = ""
    for url in probe_urls:
        try:
            response = requests.get(url, headers=headers, timeout=timeout)
            if response.status_code < 500:
                return True, url
            last_error = f"{url} -> HTTP {response.status_code}"
        except requests.RequestException as exc:
            last_error = f"{url} -> {type(exc).__name__}: {exc}"
    return False, last_error


def get_eval_endpoint_or_none(verbose: bool = True) -> Optional[str]:
    global OPENAI_API_BASE, vllm_process

    ready, detail = check_openai_compatible_endpoint(OPENAI_API_BASE, OPENAI_API_KEY)
    if ready:
        if verbose:
            print(f"Evaluation endpoint is ready: {detail}")
        return OPENAI_API_BASE

    local_prefixes = (
        "http://localhost",
        "http://127.0.0.1",
        "https://localhost",
        "https://127.0.0.1",
    )
    points_to_localhost = OPENAI_API_BASE.startswith(local_prefixes)

    if points_to_localhost and can_attempt_local_vllm():
        if vllm_process is None:
            if verbose:
                print("No local OpenAI-compatible endpoint detected. Starting local vLLM for evaluation...")
            vllm_process = launch_local_vllm()
        OPENAI_API_BASE = "http://localhost:9999/v1"
        ready, detail = check_openai_compatible_endpoint(OPENAI_API_BASE, OPENAI_API_KEY)
        if ready:
            if verbose:
                print(f"Local vLLM endpoint is ready: {detail}")
            return OPENAI_API_BASE

    if verbose:
        log_tail = "\n".join(last_vllm_logs[-10:]) if last_vllm_logs else "(no vLLM logs captured)"
        if points_to_localhost:
            if running_in_colab():
                print(
                    "Colab environment detected and no usable local OpenAI-compatible endpoint is available. "
                    "RAGAS and LLM-based evaluation will be skipped unless you set external OPENAI_API_BASE / OPENAI_API_KEY."
                )
            else:
                print(
                    "No usable local OpenAI-compatible endpoint is available. "
                    "Install vllm or set external OPENAI_API_BASE / OPENAI_API_KEY to enable RAGAS."
                )
        else:
            print(
                "Configured OPENAI_API_BASE is not reachable. "
                "RAGAS and LLM-based evaluation will be skipped unless the endpoint becomes available."
            )
        print(f"Last endpoint error: {detail}")
        print(f"Recent vLLM logs: {log_tail}")

    return None


def ensure_eval_endpoint() -> str:
    endpoint = get_eval_endpoint_or_none(verbose=True)
    if endpoint is not None:
        return endpoint
    raise RuntimeError(
        "Не удалось подключиться к OpenAI-compatible endpoint для evaluation. "
        "Укажите внешний OPENAI_API_BASE / OPENAI_API_KEY или установите vllm в среде с CUDA."
    )


class LocalLLM:
    def __init__(
        self,
        model_name: str = EVAL_MODEL_NAME,
        api_base: str = OPENAI_API_BASE,
        api_key: str = OPENAI_API_KEY,
        temperature: float = 0.1,
        max_tokens: int = 256,
    ):
        self.model_name = model_name
        self.api_key = api_key
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.api_url = api_base.rstrip("/") + "/chat/completions"

    def get_response(self, prompt: str, system_prompt: str = "") -> str:
        payload = {
            "model": self.model_name,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt},
            ],
            "temperature": self.temperature,
            "max_tokens": self.max_tokens,
        }
        headers = {"Content-Type": "application/json"}
        if self.api_key and self.api_key != "EMPTY":
            headers["Authorization"] = f"Bearer {self.api_key}"
        response = requests.post(self.api_url, headers=headers, json=payload, timeout=180)
        response.raise_for_status()
        return response.json()["choices"][0]["message"]["content"].strip()


vllm_process = None
if START_LOCAL_VLLM:
    if can_attempt_local_vllm():
        vllm_process = launch_local_vllm()
        OPENAI_API_BASE = "http://localhost:9999/v1"
    else:
        print("START_LOCAL_VLLM=True, but local vLLM prerequisites are not available. Startup skipped.")


In [ ]:
class AdvancedRAGPipeline:
    def __init__(
        self,
        spec: ModelSpec,
        reranker_name: str = RERANKER_NAME,
        embedding_device: str = EMBEDDING_DEVICE,
        reranker_device: str = RERANKER_DEVICE,
    ):
        self.spec = spec
        self.embedding_device = embedding_device
        self.reranker_device = reranker_device
        self.reranker_name = reranker_name

        print(f"Loading embedding model: {spec.model_name} on {embedding_device}")
        self.embedding_model = SentenceTransformer(spec.model_name, device=embedding_device)
        self.embedding_model.max_seq_length = min(spec.max_length, self.embedding_model.max_seq_length)

        print(f"Loading reranker: {reranker_name} on {reranker_device}")
        self.cross_encoder = CrossEncoder(reranker_name, device=reranker_device)

        self.chunks: List[Dict[str, Any]] = []
        self.bm25: Optional[Any] = None
        self.bm25_backend = "uninitialized"
        self.index: Optional[Any] = None
        self.index_backend = "uninitialized"

    def process_documents(self, documents: List[Tuple[str, Dict[str, Any]]]) -> None:
        self.chunks = [{"text": text, "metadata": metadata} for text, metadata in documents]
        tokenized_corpus = [tokenize_ru_medical(chunk["text"]) for chunk in self.chunks]
        self.bm25, self.bm25_backend = build_lexical_retriever(tokenized_corpus)

        embeddings = self.embedding_model.encode(
            [chunk["text"] for chunk in self.chunks],
            batch_size=self.spec.batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).astype("float32")

        self.index, self.index_backend = build_dense_index(embeddings)
        print(
            f"Indexed {len(self.chunks)} case documents "
            f"(lexical_backend={self.bm25_backend}, dense_backend={self.index_backend})"
        )

    def search(self, query: str, dense_top_k: int = DENSE_TOP_K, final_top_k: int = FINAL_TOP_K) -> List[Dict[str, Any]]:
        if self.bm25 is None or self.index is None:
            raise RuntimeError("Documents are not indexed. Call process_documents() first.")

        tokenized_query = tokenize_ru_medical(query)
        bm25_scores = self.bm25.get_scores(tokenized_query)
        bm25_idx = np.argsort(bm25_scores)[::-1][:dense_top_k]

        query_embedding = self.embedding_model.encode(
            [query],
            batch_size=1,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).astype("float32")
        _, dense_idx = self.index.search(query_embedding, dense_top_k)

        candidate_indices = list(dict.fromkeys(list(bm25_idx) + list(dense_idx[0])))
        candidate_chunks = [self.chunks[i] for i in candidate_indices]
        cross_inputs = [[query, item["text"]] for item in candidate_chunks]
        cross_scores = self.cross_encoder.predict(cross_inputs, batch_size=min(16, len(cross_inputs)))

        ranked = sorted(
            zip(candidate_chunks, cross_scores),
            key=lambda pair: float(pair[1]),
            reverse=True,
        )

        final_items: List[Dict[str, Any]] = []
        for rank, (chunk, score) in enumerate(ranked[:final_top_k], start=1):
            final_items.append(
                {
                    "rank": rank,
                    "text": chunk["text"],
                    "metadata": chunk["metadata"],
                    "reranker_score": float(score),
                }
            )
        return final_items

    @staticmethod
    def build_context_block(retrieved: List[Dict[str, Any]]) -> str:
        blocks = []
        for item in retrieved:
            case_id = item["metadata"]["case_id"]
            blocks.append(f"[Case {case_id} | rank={item['rank']}] {item['text']}")
        return "\n\n---\n\n".join(blocks)

    def answer_query(self, query: str, llm: LocalLLM, retrieved: Optional[List[Dict[str, Any]]] = None) -> Tuple[str, List[Dict[str, Any]]]:
        retrieved = retrieved or self.search(query)
        context = self.build_context_block(retrieved)
        system_prompt = (
            "Ты опытный врач-цитолог. Отвечай только на основе предоставленного контекста. "
            "Сначала назови диагностическую категорию Bethesda, затем дай одно короткое обоснование."
        )
        user_prompt = f"Контекст:\n{context}\n\nВопрос: {query}\nОтвет:"
        return llm.get_response(user_prompt, system_prompt), retrieved

    def cleanup(self) -> None:
        for attr in ["embedding_model", "cross_encoder", "bm25", "index", "chunks"]:
            if hasattr(self, attr):
                setattr(self, attr, None)
        clear_torch_memory()


In [ ]:
class AdvancedRAGEvaluator:
    def __init__(self, llm: Optional[LocalLLM] = None):
        self.llm = llm

    @staticmethod
    def parse_binary_judge(text: str) -> float:
        match = re.search(r"\b([01])\b", text)
        if match:
            return float(match.group(1))
        return np.nan

    def judge_binary(self, prompt: str) -> float:
        if self.llm is None:
            return np.nan
        raw = self.llm.get_response(prompt, system_prompt="Верни только 1 или 0.")
        return self.parse_binary_judge(raw)

    def generate_predictions(self, retrieval_rows: List[Dict[str, Any]]) -> pd.DataFrame:
        if self.llm is None:
            raise RuntimeError("Для генерации ответов нужен LocalLLM или совместимый endpoint.")

        records: List[Dict[str, Any]] = []
        for row in retrieval_rows:
            contexts = row["retrieved_contexts"]
            joined_context = "\n\n---\n\n".join(contexts)
            question = row["retrieval_query"]
            system_prompt = (
                "Ты опытный врач-цитолог. Используй только контекст. "
                "Верни категорию Bethesda и краткое обоснование."
            )
            user_prompt = f"Контекст:\n{joined_context}\n\nВопрос: {question}\nОтвет:"
            answer = self.llm.get_response(user_prompt, system_prompt=system_prompt)
            records.append(
                {
                    **row,
                    "question": question,
                    "answer": answer,
                }
            )
        return pd.DataFrame(records)

    def evaluate_manual(self, prediction_df: pd.DataFrame) -> Tuple[Dict[str, Any], pd.DataFrame]:
        judged_df = prediction_df.copy()
        judged_df["hit_rate"] = judged_df["gold_case_hit"].astype(float)

        if self.llm is None:
            summary = {
                "model": judged_df["model"].iloc[0],
                "hit_rate": round(judged_df["hit_rate"].mean() * 100, 2),
                "context_relevance": np.nan,
                "faithfulness": np.nan,
                "answer_correctness": np.nan,
            }
            return summary, judged_df

        context_relevance_scores = []
        faithfulness_scores = []
        correctness_scores = []

        for _, row in judged_df.iterrows():
            context_text = "\n\n---\n\n".join(row["retrieved_contexts"])

            rel_prompt = (
                f"Контекст:\n{context_text}\n\n"
                f"Вопрос: {row['question']}\n"
                "Содержит ли найденный контекст достаточно информации, чтобы корректно определить категорию Bethesda?"
            )
            context_relevance_scores.append(self.judge_binary(rel_prompt))

            faith_prompt = (
                f"Контекст:\n{context_text}\n\n"
                f"Ответ: {row['answer']}\n"
                "Строго ли ответ основан на контексте без новых фактов?"
            )
            faithfulness_scores.append(self.judge_binary(faith_prompt))

            corr_prompt = (
                f"Вопрос: {row['question']}\n"
                f"Эталон: {row['ground_truth']}\n"
                f"Генерация: {row['answer']}\n"
                "Фактически корректен ли ответ относительно эталона?"
            )
            correctness_scores.append(self.judge_binary(corr_prompt))

        judged_df["context_relevance"] = context_relevance_scores
        judged_df["faithfulness_manual"] = faithfulness_scores
        judged_df["answer_correctness"] = correctness_scores

        summary = {
            "model": judged_df["model"].iloc[0],
            "hit_rate": round(judged_df["hit_rate"].mean() * 100, 2),
            "context_relevance": round(judged_df["context_relevance"].mean() * 100, 2),
            "faithfulness": round(judged_df["faithfulness_manual"].mean() * 100, 2),
            "answer_correctness": round(judged_df["answer_correctness"].mean() * 100, 2),
        }
        return summary, judged_df


In [ ]:
retrieval_summary_df = pd.DataFrame()
retrieval_payload: Dict[str, Any] = {}


def run_retrieval_sweep(model_specs: List[ModelSpec]) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    summary_rows: List[Dict[str, Any]] = []
    payload: Dict[str, Any] = {}

    for spec in model_specs:
        print("=" * 80)
        print(f"Running retrieval sweep for {spec.label} -> {spec.model_name}")
        pipeline = None
        try:
            clear_torch_memory()
            pipeline = AdvancedRAGPipeline(spec)
            pipeline.process_documents(documents)

            model_rows: List[Dict[str, Any]] = []
            reciprocal_ranks: List[float] = []

            for row in eval_rows:
                retrieved = pipeline.search(row["retrieval_query"])
                retrieved_ids = [item["metadata"]["case_id"] for item in retrieved]
                retrieved_contexts = [item["text"] for item in retrieved]
                rank = next((idx + 1 for idx, case_id in enumerate(retrieved_ids) if case_id == row["id"]), None)
                reciprocal_ranks.append(0.0 if rank is None else 1.0 / rank)

                model_rows.append(
                    {
                        "model": spec.label,
                        "hf_model": spec.model_name,
                        "query_id": row["id"],
                        "question": row["retrieval_query"],
                        "retrieval_query": row["retrieval_query"],
                        "case_anchor": row["case_anchor"],
                        "ground_truth": row["ground_truth"],
                        "bethesda_label": row["bethesda_label"],
                        "gold_case_id": row["id"],
                        "retrieved_case_ids": retrieved_ids,
                        "retrieved_contexts": retrieved_contexts,
                        "gold_case_hit": int(row["id"] in retrieved_ids),
                        "gold_case_rank": rank,
                    }
                )

            summary = {
                "model": spec.label,
                "hf_model": spec.model_name,
                "status": "ok",
                "hit_rate": round(np.mean([row["gold_case_hit"] for row in model_rows]) * 100, 2),
                "mrr": round(np.mean(reciprocal_ranks), 4),
                "batch_size": spec.batch_size,
                "max_length": spec.max_length,
                "notes": spec.notes,
            }
            summary_rows.append(summary)
            payload[spec.label] = {
                "status": "ok",
                "spec": asdict(spec),
                "rows": model_rows,
            }
        except Exception as exc:
            summary_rows.append(
                {
                    "model": spec.label,
                    "hf_model": spec.model_name,
                    "status": "failed",
                    "hit_rate": np.nan,
                    "mrr": np.nan,
                    "batch_size": spec.batch_size,
                    "max_length": spec.max_length,
                    "notes": f"{spec.notes} | error={type(exc).__name__}: {exc}",
                }
            )
            payload[spec.label] = {
                "status": "failed",
                "spec": asdict(spec),
                "error": f"{type(exc).__name__}: {exc}",
                "rows": [],
            }
        finally:
            if pipeline is not None:
                pipeline.cleanup()

    summary_df = pd.DataFrame(summary_rows).sort_values(["status", "hit_rate"], ascending=[True, False])
    return summary_df, payload


if RUN_RETRIEVAL_SWEEP:
    retrieval_summary_df, retrieval_payload = run_retrieval_sweep(MODEL_SPECS)
    retrieval_summary_df.to_csv(ARTIFACT_DIR / "retrieval_summary.csv", index=False, encoding="utf-8-sig")
    (ARTIFACT_DIR / "retrieval_payload.json").write_text(
        json.dumps(retrieval_payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    display(retrieval_summary_df)
else:
    payload_path = ARTIFACT_DIR / "retrieval_payload.json"
    if payload_path.exists():
        retrieval_payload = json.loads(payload_path.read_text(encoding="utf-8"))
        retrieval_summary_df = pd.read_csv(ARTIFACT_DIR / "retrieval_summary.csv")
        display(retrieval_summary_df)
    else:
        print("Retrieval sweep skipped and no cached payload found.")


In [ ]:
manual_summary_df = pd.DataFrame()
prediction_frames: Dict[str, pd.DataFrame] = {}
manual_detail_frames: Dict[str, pd.DataFrame] = {}

if RUN_LLM_MANUAL_METRICS:
    eval_api_base = ensure_eval_endpoint()
    llm = LocalLLM(model_name=EVAL_MODEL_NAME, api_base=eval_api_base, api_key=OPENAI_API_KEY)
    evaluator = AdvancedRAGEvaluator(llm=llm)
    manual_rows: List[Dict[str, Any]] = []

    for spec in MODEL_SPECS:
        payload = retrieval_payload.get(spec.label, {})
        if payload.get("status") != "ok":
            continue
        prediction_df = evaluator.generate_predictions(payload["rows"])
        summary, judged_df = evaluator.evaluate_manual(prediction_df)
        prediction_frames[spec.label] = prediction_df
        manual_detail_frames[spec.label] = judged_df
        manual_rows.append(summary)

    manual_summary_df = pd.DataFrame(manual_rows)
    if not manual_summary_df.empty:
        manual_summary_df = manual_summary_df.sort_values("hit_rate", ascending=False)
        manual_summary_df.to_csv(ARTIFACT_DIR / "manual_metrics_summary.csv", index=False, encoding="utf-8-sig")
        display(manual_summary_df)
    else:
        print("Manual evaluation was enabled, but no successful model runs were available.")

    if manual_detail_frames:
        pd.concat(manual_detail_frames.values(), ignore_index=True).to_json(
            ARTIFACT_DIR / "manual_metrics_details.json",
            orient="records",
            force_ascii=False,
            indent=2,
        )
else:
    print("RUN_LLM_MANUAL_METRICS=False -> manual LLM-judge metrics skipped.")


In [ ]:
RAGAS_DEPENDENCIES_READY = pip_install_if_needed(
    packages=[
        "datasets>=4,<5",
        "ragas>=0.4,<0.5",
        "langchain-openai",
        "langchain-huggingface",
    ],
    required_modules=[
        "datasets",
        "ragas",
        "langchain_openai",
        "langchain_huggingface",
    ],
    label="RAGAS dependencies",
    optional=True,
)

ragas_summary_df = pd.DataFrame()
ragas_detail_frames: Dict[str, pd.DataFrame] = {}
RAGAS_IMPORTS_READY = False
RAGAS_IMPORT_ERROR: Optional[str] = None

try:
    from ragas import evaluate
    from ragas.dataset_schema import EvaluationDataset
    from ragas.metrics import answer_relevancy, context_precision, context_recall, faithfulness
    from ragas.run_config import RunConfig
    from langchain_huggingface import HuggingFaceEmbeddings
    from langchain_openai import ChatOpenAI
    RAGAS_IMPORTS_READY = True
except Exception as exc:
    RAGAS_IMPORT_ERROR = f"{type(exc).__name__}: {exc}"
    print("RAGAS imports are unavailable, so RAGAS evaluation will be skipped.")
    print(f"  Import error: {RAGAS_IMPORT_ERROR}")
    print(f"  Install status: {MODULE_INSTALL_STATUS.get('RAGAS dependencies')}")
    RUN_RAGAS = False


if RAGAS_IMPORTS_READY:
    def safe_metric_mean(frame: pd.DataFrame, column: str) -> float:
        if column not in frame.columns:
            return np.nan
        series = pd.to_numeric(frame[column], errors="coerce")
        return round(series.mean() * 100, 2) if series.notna().any() else np.nan


    def build_prediction_frame_if_needed(model_label: str, rows: List[Dict[str, Any]], llm: LocalLLM) -> pd.DataFrame:
        if model_label in prediction_frames:
            return prediction_frames[model_label]
        evaluator = AdvancedRAGEvaluator(llm=llm)
        prediction_df = evaluator.generate_predictions(rows)
        prediction_frames[model_label] = prediction_df
        return prediction_df


    def coerce_text(value: Any) -> str:
        if value is None:
            return ""
        if isinstance(value, float) and np.isnan(value):
            return ""
        return str(value).strip()


    def normalize_ragas_contexts(value: Any) -> List[str]:
        if isinstance(value, list):
            return [str(item).strip() for item in value if item is not None and not (isinstance(item, float) and np.isnan(item))]
        if value is None:
            return []
        if isinstance(value, float) and np.isnan(value):
            return []
        return [str(value).strip()]


    def prediction_frame_to_ragas_dataset(prediction_df: pd.DataFrame) -> EvaluationDataset:
        records: List[Dict[str, Any]] = []
        for row in prediction_df.to_dict(orient="records"):
            records.append(
                {
                    "user_input": coerce_text(row.get("question") or row.get("retrieval_query")),
                    "response": coerce_text(row.get("answer")),
                    "retrieved_contexts": normalize_ragas_contexts(row.get("retrieved_contexts")),
                    "reference": coerce_text(row.get("ground_truth")),
                }
            )
        return EvaluationDataset.from_list(records)


    def evaluate_with_ragas(model_label: str, model_name: str, prediction_df: pd.DataFrame) -> Tuple[Dict[str, Any], pd.DataFrame]:
        eval_api_base = ensure_eval_endpoint()
        ragas_llm = ChatOpenAI(
            model=EVAL_MODEL_NAME,
            api_key=OPENAI_API_KEY,
            base_url=eval_api_base,
            temperature=0.1,
            max_completion_tokens=384,
        )

        ragas_embeddings = HuggingFaceEmbeddings(
            model_name=model_name,
            model_kwargs={"device": "cpu"},
            encode_kwargs={"normalize_embeddings": True},
        )

        dataset = prediction_frame_to_ragas_dataset(prediction_df)
        result = evaluate(
            dataset=dataset,
            metrics=[context_precision, context_recall, faithfulness, answer_relevancy],
            llm=ragas_llm,
            embeddings=ragas_embeddings,
            run_config=RunConfig(max_workers=1, timeout=180),
            raise_exceptions=False,
        )
        df = result.to_pandas()
        summary = {
            "model": model_label,
            "context_precision": safe_metric_mean(df, "context_precision"),
            "context_recall": safe_metric_mean(df, "context_recall"),
            "faithfulness_ragas": safe_metric_mean(df, "faithfulness"),
            "answer_relevancy": safe_metric_mean(df, "answer_relevancy"),
        }
        return summary, df


    eval_api_base = get_eval_endpoint_or_none(verbose=RUN_RAGAS)
    if RUN_RAGAS and eval_api_base is not None:
        llm = LocalLLM(model_name=EVAL_MODEL_NAME, api_base=eval_api_base, api_key=OPENAI_API_KEY)
        ragas_rows: List[Dict[str, Any]] = []

        for spec in MODEL_SPECS:
            payload = retrieval_payload.get(spec.label, {})
            if payload.get("status") != "ok":
                continue
            prediction_df = build_prediction_frame_if_needed(spec.label, payload["rows"], llm)
            summary, detail_df = evaluate_with_ragas(spec.label, spec.model_name, prediction_df)
            ragas_rows.append(summary)
            ragas_detail_frames[spec.label] = detail_df

        ragas_summary_df = pd.DataFrame(ragas_rows)
        if not ragas_summary_df.empty:
            ragas_summary_df = ragas_summary_df.sort_values("context_precision", ascending=False)
            ragas_summary_df.to_csv(ARTIFACT_DIR / "ragas_metrics_summary.csv", index=False, encoding="utf-8-sig")
            display(ragas_summary_df)
        else:
            print("Ragas was enabled, but no successful model runs were available.")
            failed_models = [spec.label for spec in MODEL_SPECS if retrieval_payload.get(spec.label, {}).get("status") != "ok"]
            if failed_models:
                print(f"  Models skipped because retrieval did not finish successfully: {failed_models}")
            else:
                print("  Retrieval payload is empty. Re-run the retrieval sweep cell above and check its output.")
    elif RUN_RAGAS:
        print("RUN_RAGAS=True, but no evaluation endpoint is available.")
        explain_eval_skip("RAGAS evaluation", api_base=OPENAI_API_BASE)
    else:
        print("RUN_RAGAS=False -> ragas evaluation skipped.")


In [ ]:
comparison_df = retrieval_summary_df[["model", "hf_model", "hit_rate", "mrr", "status"]].copy()
comparison_df = comparison_df.rename(columns={"hit_rate": "hit_rate_retrieval"})

if not manual_summary_df.empty:
    comparison_df = comparison_df.merge(manual_summary_df, on="model", how="left")

if not ragas_summary_df.empty:
    comparison_df = comparison_df.merge(ragas_summary_df, on="model", how="left")

comparison_path = ARTIFACT_DIR / "combined_metrics.csv"
comparison_df.to_csv(comparison_path, index=False, encoding="utf-8-sig")

display(comparison_df)
print(f"Combined metrics saved to: {comparison_path}")

if not existing_ragas_baseline_df.empty:
    print("Existing project-level ragas baseline from ragas_evaluation_results.csv")
    display(existing_ragas_baseline_df)


## Chunking Strategy Study

Goal:
- Reuse the retrieval and `ragas` stack from the embedding sweep, but keep the embedding backbone fixed.
- Compare several chunk construction strategies that are realistic for short Russian cytology reports.
- Save report-friendly markdown artifacts so the results can be copied directly into a presentation or `.md` file.

Strategies in this section:
- `whole_case`: baseline, one case equals one document.
- `sentence_window_2`: sliding windows of two sentences with overlap.
- `structural_subcase`: split by subcase or lobe headers such as `Пр. доля`, `Лев. доля`, `П 17`, `Л 17`.
- `structural_then_sentence_window_2`: structural split first, then short sentence windows inside long subcases.


In [ ]:
@dataclass
class ChunkingSpec:
    label: str
    notes: str


CHUNKING_STRATEGIES: List[ChunkingSpec] = [
    ChunkingSpec(
        label="whole_case",
        notes="Baseline: one cytology case per indexed document",
    ),
    ChunkingSpec(
        label="sentence_window_2",
        notes="Sliding 2-sentence windows with overlap=1",
    ),
    ChunkingSpec(
        label="structural_subcase",
        notes="Split by lobe or subcase markers when present",
    ),
    ChunkingSpec(
        label="structural_then_sentence_window_2",
        notes="Split by subcase first, then 2-sentence windows inside each segment",
    ),
]

CHUNKING_BACKBONE_LABEL = os.getenv("CHUNKING_BACKBONE_LABEL", "bge_m3")
CHUNKING_ARTIFACT_DIR = ARTIFACT_DIR / "chunking_study"
CHUNKING_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


def resolve_model_spec(label: str, specs: List[ModelSpec]) -> ModelSpec:
    for spec in specs:
        if spec.label == label:
            return spec
    available = ", ".join(spec.label for spec in specs)
    raise ValueError(f"Unknown model label: {label}. Available: {available}")


CHUNKING_MODEL_SPEC = resolve_model_spec(CHUNKING_BACKBONE_LABEL, MODEL_SPECS)


def split_sentences_ru(text: str) -> List[str]:
    sentences = [part.strip() for part in re.split(r"(?<=[.!?])\s+", text.strip()) if part.strip()]
    return sentences or [text.strip()]


def is_subcase_header(sentence: str) -> bool:
    header_markers = (
        "пр. доля",
        "лев. доля",
        "п ",
        "л ",
        "п/п",
        "н/3",
        "прав.",
        "лев.",
    )
    sentence_lc = sentence.lower()
    if "№" not in sentence and not any(sentence_lc.startswith(marker) for marker in header_markers):
        return False
    if len(sentence) > 90 and sentence.count(".") < 2:
        return False
    return any(sentence_lc.startswith(marker) for marker in header_markers) or "№" in sentence


def split_structural_subcases(text: str) -> List[str]:
    sentences = split_sentences_ru(text)
    segments: List[List[str]] = []
    current: List[str] = []

    for sentence in sentences:
        if is_subcase_header(sentence) and current:
            segments.append(current)
            current = [sentence]
        else:
            current.append(sentence)

    if current:
        segments.append(current)

    return [" ".join(segment).strip() for segment in segments if " ".join(segment).strip()]


def build_sentence_windows(sentences: List[str], window_size: int = 2, overlap: int = 1) -> List[str]:
    if len(sentences) <= window_size:
        return [" ".join(sentences).strip()]

    step = max(1, window_size - overlap)
    windows: List[str] = []
    for start in range(0, len(sentences), step):
        window = sentences[start : start + window_size]
        if not window:
            continue
        windows.append(" ".join(window).strip())
        if start + window_size >= len(sentences):
            break
    return windows


def make_chunk_record(
    text: str,
    row: Dict[str, Any],
    chunking_label: str,
    chunk_index: int,
    parent_segment_index: int = 0,
) -> Tuple[str, Dict[str, Any]]:
    return (
        text,
        {
            "case_id": row["id"],
            "case_anchor": row["case_anchor"],
            "chunking": chunking_label,
            "chunk_id": f"{row['id']}::{chunking_label}::{chunk_index}",
            "parent_segment_index": parent_segment_index,
        },
    )


def build_chunked_documents(rows: List[Dict[str, Any]], spec: ChunkingSpec) -> List[Tuple[str, Dict[str, Any]]]:
    documents: List[Tuple[str, Dict[str, Any]]] = []

    for row in rows:
        text = row["simulated_context"]

        if spec.label == "whole_case":
            documents.append(make_chunk_record(text, row, spec.label, chunk_index=0))
            continue

        if spec.label == "sentence_window_2":
            windows = build_sentence_windows(split_sentences_ru(text), window_size=2, overlap=1)
            for idx, window in enumerate(windows):
                documents.append(make_chunk_record(window, row, spec.label, chunk_index=idx))
            continue

        if spec.label == "structural_subcase":
            segments = split_structural_subcases(text)
            for idx, segment in enumerate(segments):
                documents.append(
                    make_chunk_record(segment, row, spec.label, chunk_index=idx, parent_segment_index=idx)
                )
            continue

        if spec.label == "structural_then_sentence_window_2":
            segments = split_structural_subcases(text)
            chunk_index = 0
            for segment_idx, segment in enumerate(segments):
                windows = build_sentence_windows(split_sentences_ru(segment), window_size=2, overlap=1)
                for window in windows:
                    documents.append(
                        make_chunk_record(
                            window,
                            row,
                            spec.label,
                            chunk_index=chunk_index,
                            parent_segment_index=segment_idx,
                        )
                    )
                    chunk_index += 1
            continue

        raise ValueError(f"Unsupported chunking strategy: {spec.label}")

    return documents


def dataframe_to_markdown(frame: pd.DataFrame, columns: Optional[List[str]] = None) -> str:
    if frame.empty:
        return "_No rows to display._"

    view = frame.copy()
    if columns is not None:
        view = view[columns]

    def normalize_value(value: Any) -> str:
        if value is None:
            return ""
        if isinstance(value, float):
            if np.isnan(value):
                return ""
            return f"{value:.2f}"
        return str(value)

    headers = list(view.columns)
    rows = [[normalize_value(value) for value in row] for row in view.to_numpy().tolist()]
    widths = [len(header) for header in headers]
    for row in rows:
        widths = [max(width, len(cell)) for width, cell in zip(widths, row)]

    def format_row(row: List[str]) -> str:
        return "| " + " | ".join(cell.ljust(width) for cell, width in zip(row, widths)) + " |"

    separator = "| " + " | ".join("-" * width for width in widths) + " |"
    lines = [format_row(headers), separator]
    lines.extend(format_row(row) for row in rows)
    return "\n".join(lines)


chunking_preview_rows: List[Dict[str, Any]] = []
for spec in CHUNKING_STRATEGIES:
    docs = build_chunked_documents(eval_rows, spec)
    chunking_preview_rows.append(
        {
            "chunking": spec.label,
            "chunks_total": len(docs),
            "avg_chunks_per_case": round(len(docs) / len(eval_rows), 2),
            "notes": spec.notes,
        }
    )

chunking_preview_df = pd.DataFrame(chunking_preview_rows)
display(chunking_preview_df)
print(
    {
        "chunking_backbone_label": CHUNKING_MODEL_SPEC.label,
        "chunking_backbone_model": CHUNKING_MODEL_SPEC.model_name,
        "chunking_artifacts": str(CHUNKING_ARTIFACT_DIR),
    }
)


In [ ]:
chunking_retrieval_summary_df = pd.DataFrame()
chunking_retrieval_payload: Dict[str, Any] = {}


def run_chunking_sweep(
    chunking_specs: List[ChunkingSpec],
    embedding_spec: ModelSpec,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    summary_rows: List[Dict[str, Any]] = []
    payload: Dict[str, Any] = {}

    for chunk_spec in chunking_specs:
        print("=" * 80)
        print(
            f"Running chunking sweep for {chunk_spec.label} using "
            f"{embedding_spec.label} -> {embedding_spec.model_name}"
        )
        pipeline = None
        chunked_documents = build_chunked_documents(eval_rows, chunk_spec)

        try:
            clear_torch_memory()
            pipeline = AdvancedRAGPipeline(embedding_spec)
            pipeline.process_documents(chunked_documents)

            model_rows: List[Dict[str, Any]] = []
            reciprocal_ranks: List[float] = []

            for row in eval_rows:
                retrieved = pipeline.search(row["retrieval_query"])
                retrieved_case_ids = [item["metadata"]["case_id"] for item in retrieved]
                retrieved_chunk_ids = [item["metadata"]["chunk_id"] for item in retrieved]
                retrieved_contexts = [item["text"] for item in retrieved]
                rank = next(
                    (
                        idx + 1
                        for idx, item in enumerate(retrieved)
                        if item["metadata"]["case_id"] == row["id"]
                    ),
                    None,
                )
                reciprocal_ranks.append(0.0 if rank is None else 1.0 / rank)

                model_rows.append(
                    {
                        "chunking": chunk_spec.label,
                        "hf_model": embedding_spec.model_name,
                        "query_id": row["id"],
                        "question": row["retrieval_query"],
                        "retrieval_query": row["retrieval_query"],
                        "case_anchor": row["case_anchor"],
                        "ground_truth": row["ground_truth"],
                        "bethesda_label": row["bethesda_label"],
                        "gold_case_id": row["id"],
                        "retrieved_case_ids": retrieved_case_ids,
                        "retrieved_chunk_ids": retrieved_chunk_ids,
                        "retrieved_contexts": retrieved_contexts,
                        "gold_case_hit": int(row["id"] in retrieved_case_ids),
                        "gold_case_rank": rank,
                    }
                )

            summary_rows.append(
                {
                    "chunking": chunk_spec.label,
                    "hf_model": embedding_spec.model_name,
                    "status": "ok",
                    "hit_rate": round(np.mean([row["gold_case_hit"] for row in model_rows]) * 100, 2),
                    "mrr": round(np.mean(reciprocal_ranks), 4),
                    "indexed_chunks": len(chunked_documents),
                    "avg_chunks_per_case": round(len(chunked_documents) / len(eval_rows), 2),
                    "notes": chunk_spec.notes,
                }
            )
            payload[chunk_spec.label] = {
                "status": "ok",
                "spec": asdict(chunk_spec),
                "embedding_spec": asdict(embedding_spec),
                "indexed_chunks": len(chunked_documents),
                "rows": model_rows,
            }
        except Exception as exc:
            summary_rows.append(
                {
                    "chunking": chunk_spec.label,
                    "hf_model": embedding_spec.model_name,
                    "status": "failed",
                    "hit_rate": np.nan,
                    "mrr": np.nan,
                    "indexed_chunks": len(chunked_documents),
                    "avg_chunks_per_case": round(len(chunked_documents) / len(eval_rows), 2),
                    "notes": f"{chunk_spec.notes} | error={type(exc).__name__}: {exc}",
                }
            )
            payload[chunk_spec.label] = {
                "status": "failed",
                "spec": asdict(chunk_spec),
                "embedding_spec": asdict(embedding_spec),
                "indexed_chunks": len(chunked_documents),
                "error": f"{type(exc).__name__}: {exc}",
                "rows": [],
            }
        finally:
            if pipeline is not None:
                pipeline.cleanup()

    summary_df = pd.DataFrame(summary_rows).sort_values(["status", "hit_rate"], ascending=[True, False])
    return summary_df, payload


if RUN_RETRIEVAL_SWEEP:
    chunking_retrieval_summary_df, chunking_retrieval_payload = run_chunking_sweep(
        CHUNKING_STRATEGIES,
        CHUNKING_MODEL_SPEC,
    )
    chunking_retrieval_summary_df.to_csv(
        CHUNKING_ARTIFACT_DIR / "chunking_retrieval_summary.csv",
        index=False,
        encoding="utf-8-sig",
    )
    (CHUNKING_ARTIFACT_DIR / "chunking_retrieval_payload.json").write_text(
        json.dumps(chunking_retrieval_payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    display(chunking_retrieval_summary_df)
else:
    chunking_payload_path = CHUNKING_ARTIFACT_DIR / "chunking_retrieval_payload.json"
    chunking_summary_path = CHUNKING_ARTIFACT_DIR / "chunking_retrieval_summary.csv"
    if chunking_payload_path.exists() and chunking_summary_path.exists():
        chunking_retrieval_payload = json.loads(chunking_payload_path.read_text(encoding="utf-8"))
        chunking_retrieval_summary_df = pd.read_csv(chunking_summary_path)
        display(chunking_retrieval_summary_df)
    else:
        print("Chunking retrieval sweep skipped and no cached payload found.")


In [ ]:
chunking_prediction_frames: Dict[str, pd.DataFrame] = {}
chunking_ragas_summary_df = pd.DataFrame()
chunking_ragas_detail_frames: Dict[str, pd.DataFrame] = {}


def build_chunking_prediction_frame_if_needed(
    chunking_label: str,
    rows: List[Dict[str, Any]],
    llm: LocalLLM,
) -> pd.DataFrame:
    if chunking_label in chunking_prediction_frames:
        return chunking_prediction_frames[chunking_label]
    evaluator = AdvancedRAGEvaluator(llm=llm)
    prediction_df = evaluator.generate_predictions(rows)
    chunking_prediction_frames[chunking_label] = prediction_df
    return prediction_df


eval_api_base = get_eval_endpoint_or_none(verbose=RUN_RAGAS)
if RUN_RAGAS and eval_api_base is not None:
    llm = LocalLLM(model_name=EVAL_MODEL_NAME, api_base=eval_api_base, api_key=OPENAI_API_KEY)
    ragas_rows: List[Dict[str, Any]] = []

    for chunk_spec in CHUNKING_STRATEGIES:
        payload = chunking_retrieval_payload.get(chunk_spec.label, {})
        if payload.get("status") != "ok":
            continue

        prediction_df = build_chunking_prediction_frame_if_needed(chunk_spec.label, payload["rows"], llm)
        summary, detail_df = evaluate_with_ragas(
            model_label=chunk_spec.label,
            model_name=CHUNKING_MODEL_SPEC.model_name,
            prediction_df=prediction_df,
        )
        summary["chunking"] = summary.pop("model")
        summary["hf_model"] = CHUNKING_MODEL_SPEC.model_name
        summary["notes"] = chunk_spec.notes
        ragas_rows.append(summary)
        chunking_ragas_detail_frames[chunk_spec.label] = detail_df

    chunking_ragas_summary_df = pd.DataFrame(ragas_rows)
    if not chunking_ragas_summary_df.empty:
        chunking_ragas_summary_df = chunking_ragas_summary_df.sort_values(
            "context_precision",
            ascending=False,
        )
        chunking_ragas_summary_df.to_csv(
            CHUNKING_ARTIFACT_DIR / "chunking_ragas_metrics_summary.csv",
            index=False,
            encoding="utf-8-sig",
        )
        pd.concat(chunking_ragas_detail_frames, names=["chunking", "row_id"]).reset_index().to_json(
            CHUNKING_ARTIFACT_DIR / "chunking_ragas_details.json",
            orient="records",
            force_ascii=False,
            indent=2,
        )
        display(chunking_ragas_summary_df)
    else:
        print("Ragas was enabled, but no successful chunking runs were available.")
        failed_chunkings = [
            chunk_spec.label
            for chunk_spec in CHUNKING_STRATEGIES
            if chunking_retrieval_payload.get(chunk_spec.label, {}).get("status") != "ok"
        ]
        if failed_chunkings:
            print(f"  Chunking strategies skipped because retrieval did not finish successfully: {failed_chunkings}")
        else:
            print("  Chunking retrieval payload is empty. Re-run the chunking retrieval sweep cell above and check its output.")
elif RUN_RAGAS:
    print("RUN_RAGAS=True, but no evaluation endpoint is available.")
    explain_eval_skip("Chunking RAGAS evaluation", api_base=OPENAI_API_BASE)
else:
    print("RUN_RAGAS=False -> chunking ragas evaluation skipped.")


In [ ]:
chunking_comparison_df = chunking_retrieval_summary_df[
    ["chunking", "hf_model", "hit_rate", "mrr", "indexed_chunks", "avg_chunks_per_case", "status", "notes"]
].copy()
chunking_comparison_df = chunking_comparison_df.rename(columns={"hit_rate": "hit_rate_retrieval"})

if not chunking_ragas_summary_df.empty:
    chunking_comparison_df = chunking_comparison_df.merge(
        chunking_ragas_summary_df[
            ["chunking", "context_precision", "context_recall", "faithfulness_ragas", "answer_relevancy"]
        ],
        on="chunking",
        how="left",
    )

chunking_comparison_path = CHUNKING_ARTIFACT_DIR / "chunking_combined_metrics.csv"
chunking_comparison_df.to_csv(chunking_comparison_path, index=False, encoding="utf-8-sig")

report_columns = [
    "chunking",
    "hit_rate_retrieval",
    "mrr",
    "context_precision",
    "context_recall",
    "faithfulness_ragas",
    "answer_relevancy",
    "indexed_chunks",
]

available_report_columns = [column for column in report_columns if column in chunking_comparison_df.columns]
ranking_columns = [
    column
    for column in ["context_precision", "context_recall", "hit_rate_retrieval", "mrr"]
    if column in chunking_comparison_df.columns
]
ranking_orders = [False] * len(ranking_columns)
ranked_chunking_df = (
    chunking_comparison_df.sort_values(ranking_columns, ascending=ranking_orders, na_position="last")
    if ranking_columns
    else chunking_comparison_df.copy()
)

chunking_report_table = dataframe_to_markdown(
    ranked_chunking_df,
    columns=available_report_columns,
)

top_chunking_rows = ranked_chunking_df.head(2)

observations: List[str] = []
if not top_chunking_rows.empty:
    best_row = top_chunking_rows.iloc[0]
    observations.append(
        f"- Best chunking by `context_precision`: `{best_row['chunking']}` "
        f"({best_row.get('context_precision', np.nan):.2f})."
    )
    observations.append(
        f"- Same strategy reached retrieval hit rate `{best_row.get('hit_rate_retrieval', np.nan):.2f}` "
        f"with `{int(best_row.get('indexed_chunks', 0))}` indexed chunks."
    )
    if len(top_chunking_rows) > 1:
        runner_up = top_chunking_rows.iloc[1]
        observations.append(
            f"- Runner-up: `{runner_up['chunking']}` with `context_recall` "
            f"{runner_up.get('context_recall', np.nan):.2f}."
        )

report_lines = [
    "# CytoRAG Chunking Study",
    "",
    "## Configuration",
    "",
    f"- Backbone embedding label: `{CHUNKING_MODEL_SPEC.label}`",
    f"- Backbone embedding model: `{CHUNKING_MODEL_SPEC.model_name}`",
    f"- Reranker: `{RERANKER_NAME}`",
    f"- Dataset: `{DATA_PATH.name}`",
    "",
    "## Combined Metrics",
    "",
    chunking_report_table,
    "",
    "## Key Observations",
    "",
    *(observations if observations else ["- No completed runs were available."]),
    "",
    "## Artifacts",
    "",
    f"- CSV: `{(CHUNKING_ARTIFACT_DIR / 'chunking_combined_metrics.csv').name}`",
    f"- Retrieval payload: `{(CHUNKING_ARTIFACT_DIR / 'chunking_retrieval_payload.json').name}`",
    f"- RAGAS details: `{(CHUNKING_ARTIFACT_DIR / 'chunking_ragas_details.json').name}`",
]

chunking_report_path = CHUNKING_ARTIFACT_DIR / "chunking_report.md"
chunking_report_path.write_text("\n".join(report_lines), encoding="utf-8")

display(chunking_comparison_df)
print(f"Chunking combined metrics saved to: {chunking_comparison_path}")
print(f"Chunking markdown report saved to: {chunking_report_path}")
print()
print(chunking_report_path.read_text(encoding="utf-8"))


## Next Steps

- If a model fails on T4, lower `batch_size` first before changing the architecture.
- If `bge-m3` is the best retriever, consider a second experiment that uses its sparse signal instead of separate BM25.
- If structure-aware chunking wins, port the same splitter into the main CytoRAG pipeline and re-run the best embedding model on the expanded corpus.
- If the Russian models win on retrieval but biomedical models help `faithfulness`, a practical next step is a two-stage setup: Russian retriever + biomedical answer checker.
- Save the final notebook outputs after a full run so the comparison tables and markdown report remain attached to the artifact.
